# Fine-tune OCR tiếng Việt cho BetterGI (PP-OCRv5 latin rec)

**Cài đặt notebook (cột phải):** Accelerator = **GPU T4** (1 GPU là đủ), Internet = **On**.

**Chạy 2 lượt:**
1. `SMOKE_TEST = True` (mặc định) → **Run All** tương tác (~15 phút) để bắt lỗi cài đặt/đường dẫn.
2. Qua hết thì đổi `SMOKE_TEST = False` → **Save Version → Save & Run All (Commit)**. Notebook tự chạy (~2–3 giờ, tối đa 12 giờ), không cần mở trình duyệt.

**Kết quả:** tab *Output* của phiên bản → tải `vi_rec_result.zip`, gồm:
- `vi_PP-OCRv5_mobile_rec_infer/` — model mới (`slim.onnx` + `inference.yml`)
- `latin_PP-OCRv5_mobile_rec_infer/` — model latin chính thức mới nhất (dict 836) để so sánh
- `train.log`, `dict_report.txt`, `onnx_check.txt`

Sau đó chạy ở máy: `BetterGenshinImpact.OcrDataTool.exe eval --labels E:\ocr\eval_v1\labels.tsv --model <thư mục giải nén>\vi_PP-OCRv5_mobile_rec_infer --model V5Latin`

In [ ]:
import os, re, shutil, subprocess, sys, urllib.parse, urllib.request

WORK = "/kaggle/working"
FORK = "https://github.com/IreinaMaid150700/better-genshin-impact.git"
BRANCH = "feat/vi-ocr-dataset-tool"
KIT_SUBDIR = "Test/BetterGenshinImpact.OcrDataTool/training"
KIT = f"{WORK}/kit"
TRAIN_SAMPLES = 300_000
VAL_SAMPLES = 5_000
# Lần đầu để True và chạy tương tác (Run All, ~15 phút) để bắt lỗi môi trường;
# chạy qua hết thì đổi False rồi Save Version -> Save & Run All (Commit) để train thật.
SMOKE_TEST = True
# PaddleOCR chỉ nhận một cờ -o, mọi override phải nằm chung
TRAIN_OVERRIDES = []
if SMOKE_TEST:
    TRAIN_SAMPLES, VAL_SAMPLES = 3_000, 200
    TRAIN_OVERRIDES += ["Global.epoch_num=1", "'Global.eval_batch_step=[0,20]'", "Global.print_batch_step=5"]
os.chdir(WORK)


def run(cmd, cwd=WORK):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


def fetch(url, path):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(url, path)
    print(f"{path}: {os.path.getsize(path):,} bytes")


print(sys.version)
run("nvidia-smi")

## 1. Cài PaddlePaddle GPU + PaddleOCR

In [ ]:
# Bản CUDA 12.6; nếu cài lỗi do driver, đổi cu126 -> cu118
run("python -m pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ "
    "--extra-index-url https://pypi.org/simple")
run("python -c \"import paddle; print(paddle.__version__); paddle.utils.run_check()\"")
if not os.path.isdir("PaddleOCR"):
    run("git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git")
run("python -m pip install -q -r PaddleOCR/requirements.txt")
run("git -C PaddleOCR log -1 --format=%H")

## 2. Bộ train (dict, script, config) từ fork — hoặc Kaggle Dataset `vi-ocr-kit` nếu có

In [ ]:
if os.path.isdir("/kaggle/input/vi-ocr-kit"):
    shutil.copytree("/kaggle/input/vi-ocr-kit", KIT, dirs_exist_ok=True)
else:
    if not os.path.isdir("repo"):
        run(f"git clone --depth 1 --filter=blob:none --sparse -b {BRANCH} {FORK} repo")
        run(f"git sparse-checkout set {KIT_SUBDIR}", cwd=f"{WORK}/repo")
    else:
        # Chạy lại trong cùng phiên: kéo bản mới nhất của bộ train
        run(f"git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD", cwd=f"{WORK}/repo")
    shutil.copytree(f"repo/{KIT_SUBDIR}", KIT, dirs_exist_ok=True)
    run("git log -1 --format='%h %s'", cwd=f"{WORK}/repo")
print(sorted(os.listdir(KIT)))

## 3. Tải TextMap tiếng Việt, trọng số pretrained, font

In [ ]:
fetch("https://gitlab.com/Dimbreath/AnimeGameData/-/raw/master/TextMap/TextMapVI.json", f"{WORK}/data/TextMapVI.json")
fetch("https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/latin_PP-OCRv5_mobile_rec_pretrained.pdparams",
      f"{WORK}/pretrain/latin_PP-OCRv5_mobile_rec_pretrained.pdparams")
fetch("https://paddle-model-ecology.bj.bcebos.com/paddlex/official_inference_model/paddle3.0.0/latin_PP-OCRv5_mobile_rec_infer.tar",
      f"{WORK}/data/latin_infer.tar")

FONTS = [
    "alegreyasans/AlegreyaSans-Medium.ttf", "alegreyasans/AlegreyaSans-Bold.ttf",
    "philosopher/Philosopher-Regular.ttf", "philosopher/Philosopher-Bold.ttf",
    "bevietnampro/BeVietnamPro-Medium.ttf", "bevietnampro/BeVietnamPro-Bold.ttf",
    "signika/Signika[GRAD,wght].ttf", "mulish/Mulish[wght].ttf", "asap/Asap[wdth,wght].ttf",
    "notosans/NotoSans[wdth,wght].ttf", "sourcesans3/SourceSans3[wght].ttf",
    "firasans/FiraSans-Medium.ttf", "firasans/FiraSans-Bold.ttf", "opensans/OpenSans[wdth,wght].ttf",
    "nunitosans/NunitoSans[YTLC,opsz,wdth,wght].ttf", "lexend/Lexend[wght].ttf", "exo2/Exo2[wght].ttf",
    "barlow/Barlow-Medium.ttf", "barlow/Barlow-Bold.ttf",
]
for font in FONTS:
    fetch("https://raw.githubusercontent.com/google/fonts/main/ofl/" + urllib.parse.quote(font),
          f"{WORK}/fonts/" + re.sub(r"[\[\],]", "_", font.split("/")[1]))

## 4. Kiểm tra dict + mở rộng head của trọng số pretrained

In [ ]:
BASE_DICT = f"{WORK}/PaddleOCR/ppocr/utils/dict/ppocrv5_latin_dict.txt"
run(f"python {KIT}/build_vi_dict.py --base {BASE_DICT} --textmap data/TextMapVI.json "
    f"--out {KIT}/vi_dict_regen.txt --report {WORK}/dict_report.txt")
with open(f"{KIT}/vi_dict.txt", encoding="utf-8") as a, open(f"{KIT}/vi_dict_regen.txt", encoding="utf-8") as b:
    if a.read() != b.read():
        print("CẢNH BÁO: TextMap hiện tại cho ra dict khác vi_dict.txt đã chốt; vẫn dùng vi_dict.txt. Xem dict_report.txt")
run(f"python {KIT}/expand_head.py --pretrained pretrain/latin_PP-OCRv5_mobile_rec_pretrained.pdparams "
    f"--old-dict {BASE_DICT} --new-dict {KIT}/vi_dict.txt --out pretrain/vi_init.pdparams")

## 5. Sinh dữ liệu tổng hợp

In [ ]:
run(f"python {KIT}/gen_synth.py --textmap {WORK}/data/TextMapVI.json --dict {KIT}/vi_dict.txt "
    f"--fonts {WORK}/fonts --out {WORK}/synth --train {TRAIN_SAMPLES} --val {VAL_SAMPLES} --workers {os.cpu_count()}")
run(f"head -n 5 {WORK}/synth/train.txt")

## 6. Train

In [ ]:
# Batch 64 mỗi GPU (config); có 2 GPU thì train phân tán, tổng batch 128
gpu_count = len(subprocess.check_output("nvidia-smi -L", shell=True, text=True).strip().splitlines())
overrides = list(TRAIN_OVERRIDES)
if gpu_count > 1:
    overrides.append("Global.distributed=true")
    launcher = f"python -m paddle.distributed.launch --gpus {','.join(map(str, range(gpu_count)))} --log_dir {WORK}/output/launch_log"
else:
    launcher = "python"
opt = f"-o {' '.join(overrides)}" if overrides else ""
print(f"GPU: {gpu_count}")
run(f"{launcher} tools/train.py -c {KIT}/vi_PP-OCRv5_mobile_rec.yml {opt}", cwd=f"{WORK}/PaddleOCR")

## 7. Export + chuyển ONNX (giống paddle2onnx.sh của BetterGI)

In [ ]:
run(f"python tools/export_model.py -c {KIT}/vi_PP-OCRv5_mobile_rec.yml "
    f"-o Global.pretrained_model={WORK}/output/vi_rec/best_accuracy "
    f"Global.save_inference_dir={WORK}/export/vi_PP-OCRv5_mobile_rec_infer", cwd=f"{WORK}/PaddleOCR")
run(f"tar -xf {WORK}/data/latin_infer.tar -C {WORK}/export")

# paddle2onnx chỉ có bản cho Python <= 3.12 (và ghim onnx <= 1.17, không có bản cho 3.13)
# -> venv Python 3.12 riêng cho bước chuyển đổi; uv tự tải Python 3.12
P2O = f"{WORK}/p2o"
if not os.path.exists(f"{P2O}/bin/paddle2onnx"):
    paddle_version = subprocess.check_output(
        "python -c \"import paddle; print(paddle.__version__)\"", shell=True, text=True).strip().splitlines()[-1]
    run("python -m pip install -q uv")
    run(f"uv venv --python 3.12 {P2O}")
    run(f"uv pip install --python {P2O}/bin/python paddlepaddle=={paddle_version} paddle2onnx onnxslim")
run("python -m pip install -q onnxruntime")
for name in ["vi_PP-OCRv5_mobile_rec_infer", "latin_PP-OCRv5_mobile_rec_infer"]:
    run(f"{P2O}/bin/paddle2onnx --model_dir ./ --model_filename inference.json --params_filename inference.pdiparams "
        f"--save_file model.onnx && {P2O}/bin/onnxslim model.onnx slim.onnx", cwd=f"{WORK}/export/{name}")

## 8. Kiểm tra slim.onnx theo đúng tiền xử lý của BetterGI

In [ ]:
import cv2, numpy as np, onnxruntime as ort, yaml


def check(model_dir, label_file, data_dir, limit=1000):
    with open(f"{model_dir}/inference.yml", encoding="utf-8") as f:
        chars = [str(c) for c in yaml.safe_load(f)["PostProcess"]["character_dict"]]
    session = ort.InferenceSession(f"{model_dir}/slim.onnx", providers=["CPUExecutionProvider"])
    rows = [line.rstrip("\n").split("\t", 1) for line in open(label_file, encoding="utf-8")][:limit]
    correct, classes = 0, None
    for path, label in rows:
        image = cv2.imread(f"{data_dir}/{path}")  # BGR, giống BetterGI
        h, w = image.shape[:2]
        width = max(1, int(np.ceil(48 * w / h)))
        x = cv2.resize(image, (width, 48)).astype(np.float32) / 255.0
        x = ((x - 0.5) / 0.5).transpose(2, 0, 1)[None]
        out = session.run(None, {session.get_inputs()[0].name: x})[0][0]
        classes = out.shape[1]
        ids, text, last = out.argmax(1), [], 0
        for i in ids:
            if i != 0 and i != last:
                text.append(" " if i == len(chars) + 1 else chars[i - 1])
            last = i
        correct += "".join(text) == label
    return classes, len(chars), correct, len(rows)


report = []
for name in ["vi_PP-OCRv5_mobile_rec_infer", "latin_PP-OCRv5_mobile_rec_infer"]:
    classes, dict_size, correct, total = check(f"{WORK}/export/{name}", f"{WORK}/synth/val.txt", f"{WORK}/synth")
    line = f"{name}: classes={classes} dict={dict_size} (classes = dict+2: {classes == dict_size + 2}) acc_synth_val={correct}/{total}"
    print(line)
    report.append(line)
open(f"{WORK}/export/onnx_check.txt", "w", encoding="utf-8").write("\n".join(report) + "\n")

## 9. Đóng gói kết quả, dọn dữ liệu lớn

In [ ]:
for extra in [f"{WORK}/dict_report.txt", f"{WORK}/output/vi_rec/train.log"]:
    if os.path.exists(extra):
        shutil.copy(extra, f"{WORK}/export/")
shutil.make_archive(f"{WORK}/vi_rec_result", "zip", root_dir=f"{WORK}/export")
for big in ["synth", "PaddleOCR", "repo", "fonts", "data", "p2o"]:
    shutil.rmtree(f"{WORK}/{big}", ignore_errors=True)
print(os.listdir(WORK), os.path.getsize(f"{WORK}/vi_rec_result.zip"))